# H00 — 把第一份固件送进 ESP32

**学习入口：把下面 prompt 发给 VS Code Codex 侧边栏。** Notebook 也可自行逐格阅读。
C++ 在持续 ESP-IDF 工程编辑与构建；Notebook 运行仅显示示例/手动证据卡。
**状态：Alpha；固件构建与实机验收分开记录，见 ../VALIDATION.md。**

```text
开始 H00 — 把第一份固件送进 ESP32。
请先读取本工程 AGENTS.md、.academy/course.json 和 .academy/progress.md。
通过 course_root 找到 step2/course.json 和 step2/notebooks/H00-setup.ipynb。
读取 Notebook 的 cells/source 与 academy.step_id/mode；先读当前步骤相关格。
检查工程、环境和前置条件，从第一个未完成步骤开始；每轮只推进一个小步骤。
Guided 可合并指定示例并说明 diff；Modify/Debug/Build/Open 先让我尝试，按需逐级提示。
不要主动读取 solution.md 或其他答案。环境故障可以排查修复，解释后回到原步骤。
不要把挑战 Bug 自动修掉。以当前代码与真实证据验收，记录提示级别和下一步。
我已有一定 C++ 基础，Python 几乎零基础。现在开始。
```

不用 agent 时按步骤标题顺序操作；卡住后再逐级查看本课 hints.md。启动 prompt 的路径按工作区课程链接解析，不靠其他窗口或聊天记忆。

## 我们要解决什么问题？

电脑能编译 C++，为什么还不能直接把这个程序放进 ESP32？今天只完成一条链：工具链 → 固件 → Flash → 启动日志。

前置：STEP 1 可选完成；准备板和数据线。

## 本次够用的知识

芯片（Chip）、模组（Module）、开发板（Board）不是同一层。经典 ESP32-DevKitC 常带 ESP32-WROOM 模组；USB 接口通常通过桥接芯片连接 UART。target 决定编译目标，port 决定电脑连接哪台设备。桌面的可执行文件不能直接烧到 MCU；ESP-IDF 提供交叉编译器、库、bootloader、分区和工具。

先预测、运行和记录，再补概念。每节开始保存现有工程差异，后续沿用同一工程。

## 可选：Notebook 示例与证据卡

使用课程 .venv kernel 运行下格；这只是显示工具，不需要学习 Python。侧边栏学习不依赖这些格，build/flash/monitor 都在 ESP-IDF Terminal 执行。

In [ ]:
from pathlib import Path
import sys
import json

course_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "step2/course.json").is_file():
        course_root = candidate
        break
    link = candidate / ".academy/course.json"
    if link.is_file():
        course_root = Path(json.loads(link.read_text(encoding="utf-8"))["course_root"])
        break
if course_root is None or not (course_root / "step2/course.json").is_file():
    raise RuntimeError("请从课程工作区运行，或先在学习工程 .academy/course.json 链接课程根目录")
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))
from tools.esp32_notebook import install, hardware_checklist
install(get_ipython())


## 认出手中的板 · Guided

先读 ../BOARD.md，记录板上模组文字、芯片、USB 接口和官方 pinout。没有板时可以读工程和构建，但把烧录/实物项保持未确认。
暂时不接 LED、传感器或外部电源。使用一根可传数据的 USB 线，记录插入前后新增的串口。观察：供电灯亮，只能证明有电，不能证明数据链路通。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 配置 VS Code 与 ESP-IDF · Guided

安装扩展 espressif.esp-idf-extension。按 ../SETUP.md 配置 ESP-IDF v5.5.1，运行扩展 Doctor，打开 ESP-IDF Terminal 执行 idf.py --version。
Windows STEP 2 优先原生 ESP-IDF，不能把 STEP 1 的 WSL 串口假定可用。扩展名称可能随版本变化；使用官方安装流程，以 Doctor 和实际命令输出确认，不记忆按钮位置。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 打开引导工程 · Guided

如果已按 SETUP 创建并打开引导工程，直接检查文件，不重复创建。尚未创建时，在课程根目录运行 python3 tools/academy_tutor.py init-project ../guided-desk-tool --guided（Windows 用 py）。也可以把 step2/guided-device 的全部内容复制到新目录，包含隐藏文件。
在 VS Code 打开新目录；手工复制时填写 .academy/course.json 的 course_root 为实际课程路径。读 main/main.cpp，预测启动后会打印什么。这里没有 Notebook magic 依赖。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 构建和烧录 · Guided

在新工程的 ESP-IDF Terminal 运行：
```text
idf.py set-target esp32
idf.py build
idf.py -p <port> flash monitor
```
将 <port> 替换成实际串口。先检查 target 是真实芯片。build 后找 .bin、.elf、map 和 compile_commands.json。flash 前用 Ctrl+] 退出已有 monitor，避免串口占用；读启动日志，找到应用输出而不是只看下载进度。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 让固件证明它是新的 · Modify

把启动文本改成自己的设备名，重新 build/flash/monitor，观察新字符串。再仅 build、不 flash，按 Reset：板上的日志应该还是旧文本吗？
记录“编译出的文件”和“设备正在执行的文件”为什么可能不同。不要凭 VS Code 编辑器内容判断设备版本。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 三个失败属于哪一层 · Debug

只在 main.cpp 中暂时删除一处分号，运行 build，保留第一条诊断后恢复。然后用一个不存在的串口名尝试连接（不要选择其他真实设备），观察 flash/monitor 错误。
用这两份证据解释：编译器没生成固件、电脑没连接到设备、设备运行错误，应该分别看什么？本课不通过错误接线制造故障。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 写一张设备身份证 · Build

在工程 README 写出 OS、ESP-IDF/扩展版本、板型、target、port、USB 线、完整命令和日志中的应用标识。
验收：你能退出 monitor、重新接入同一台板、按 README 构建烧录；解释 bootloader 与 app 日志的区别。新聊天应从这份记录恢复，不猜串口。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 最小故障树 · Open

画出“看起来什么也没发生”的故障树，至少覆盖供电、数据线、端口占用、target、编译和应用入口。每个分支写一个可执行检查。
下一课才接外部器件；现在解释为什么只亮供电灯不足以验收。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 验收与复盘

- [ ] build 完成且记录 IDF/target
- [ ] 实际串口 flash 完成
- [ ] Reset 后出现当前应用标识

分别记录 host、firmware build、flash、日志、实物与网络往返；未知保持未确认。手动卡片只是学习者记录，不自动测量，也不自动写入进度文件。

复盘：我选择了什么机制？哪个证据排除了错误假设？哪些内容还需要测量？把当前 Step、Hint level、代码版本、证据和下次起点写入 `.academy/progress.md`。

In [ ]:
hardware_checklist(
    'H00',
    [
        'build 完成且记录 IDF/target',
        '实际串口 flash 完成',
        'Reset 后出现当前应用标识',
    ],
)


## 遇到困难时

先完成一次尝试，再打开 `step2/lessons/H00/hints.md`，每次只读当前步骤的一层。完整参考与解题分析单独保存，默认路径不主动读取。

## 官方版本参考

- [get-started/index.html](https://docs.espressif.com/projects/esp-idf/en/v5.5.1/esp32/get-started/index.html)